Test

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("cluster-test")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "pyspark-notebook")
    .getOrCreate()
)

df = spark.range(10)
df.show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



Download file

!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet -P /data

create session

spark = (
    SparkSession.builder
    .appName("yellow taxi dataframe")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "pyspark-notebook")
    .getOrCreate()
)

df = spark.read.parquet("/data/yellow_tripdata_2025-11.parquet")
df.printSchema()
df.show(5)

create temp table

In [12]:
df.registerTempTable("taxi_table")

/usr/local/spark/python/pyspark/sql/dataframe.py:330: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


run sql queries

In [28]:
spark.sql("""
SELECT count(1) FROM taxi_table
WHERE CAST(tpep_pickup_datetime AS DATE) = '2025-11-15'
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [29]:
spark.sql("""
SELECT max(tpep_dropoff_datetime - tpep_pickup_datetime) FROM taxi_table
""").show()

+---------------------------------------------------+
|max((tpep_dropoff_datetime - tpep_pickup_datetime))|
+---------------------------------------------------+
|                               INTERVAL '3 18:38...|
+---------------------------------------------------+



Optional, enforce schema

In [22]:
from pyspark.sql import types

In [24]:
df_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True),
    types.StructField("Airport_fee", types.DoubleType(), True),
    types.StructField("cbd_congestion_fee", types.DoubleType(), True),
])

In [27]:
yellow_df = spark.read.schema(df_schema).parquet("/data/yellow_tripdata_2025-11.parquet")
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    